# AI Systems 101 — Part 2: Meet the OpenAI Agents SDK

> **What this article is:** A standalone, hands-on introduction to OpenAI's official Agents SDK. By the end, you will be fluent in the SDK's core primitives — Agents, Tools, Runners, Sessions, Handoffs, Guardrails, and Tracing — and confident enough to use them in production code.
>
> **What this article assumes:** That you've either read Part 1 of this series (where we built an agent from scratch in plain Python) or you already know how the basic agent loop works: reason → act → observe → repeat → stop. If you haven't, read Part 1 first. This article makes much more sense after you've built one of these by hand.
>
> **Note:** the code snippets throughout this notebook are illustrative — they show focused concepts in isolation. The complete, runnable agent appears in Section 11. The hands-on labs at the end of this notebook will guide you through building each concept into working code yourself.



---

## Section 0 — From Scratch to SDK

In Part 1 of this series, we built a working AI agent from scratch in roughly 150 lines of plain Python. We wrote:

- An LLM call wrapper (`ask_once`, then `chat_once`)
- A Pydantic model for structured decisions (`AgentDecision`)
- A dictionary-based tool registry (`TOOLS`)
- Text descriptions of those tools in a system prompt
- A `while` loop that called the model, dispatched tools, and accumulated observations
- A `max_steps` guardrail
- A trace list for observability

It worked. And, importantly, we understood every single line of it. Nothing was hidden.

Now we're going to do something that might feel anticlimactic at first: we're going to rebuild the same agent using the **OpenAI Agents SDK** — and discover that it takes about 30 lines instead of 150.

The point of this exercise isn't to abandon what you learned. It's the opposite. Now that you understand the mechanics, you can use a production-grade framework without it feeling like magic. Every abstraction the SDK introduces will map to something you already built by hand.

> **A framework is not a replacement for understanding. It is the natural reward for it.**

```
┌─────────────────────────────────────────────────────────┐
│                  PART 1 (Notebook 1)                    │
│                                                         │
│   You wrote: messages, tool registry, decision loop,    │
│              observations, stopping conditions, trace   │
│                                                         │
│              150 lines, every line visible.             │
└─────────────────────────────────────────────────────────┘
                           │
                           ▼
┌─────────────────────────────────────────────────────────┐
│                  PART 2 (this article)                  │
│                                                         │
│   You learn: Agent, Tool, Runner, Session, Handoff,     │
│              Guardrail, Tracing — SDK primitives that   │
│              wrap exactly what you wrote.               │
│                                                         │
│              30 lines, same mechanics underneath.       │
└─────────────────────────────────────────────────────────┘
```

Let's get started.

---

## Section 1 — What the SDK Actually Is

The **OpenAI Agents SDK** is a small Python library that handles the loop and plumbing of agent execution so you can focus on agent *design*. It was released in March 2025 as the production-grade evolution of OpenAI's earlier experimental "Swarm" library, and it has since become one of the widely adopted agent frameworks in the industry.

Its design philosophy is summarised in two principles, taken directly from the official docs:

1. **Enough features to be worth using, but few enough primitives to make it quick to learn.**
2. **Works great out of the box, but you can customise exactly what happens.**

That second principle is the important one. Many agent frameworks impose a worldview — a graph DSL, a domain-specific orchestration language, an opinionated way of structuring multi-agent systems. The OpenAI SDK doesn't. It gives you a small set of well-designed Python primitives and gets out of the way.

### The Primitives

There are really only a handful of concepts you need to know to be productive:

| Primitive | What it is | What you wrote by hand in Part 1 |
|---|---|---|
| **Agent** | An LLM configured with instructions and tools | A system prompt + a list of tool functions |
| **Tool** | A Python function the agent can call | A function in your `TOOLS` dictionary |
| **Runner** | The thing that executes an agent | Your `run_agent()` `while` loop |
| **RunResult** | What the runner returns | The dictionary returned by `run_agent()` |
| **Session** | Persistent memory across turns | The `history` list passed to `chat_once()` |
| **Handoff** | An agent transferring control to another agent | (You didn't build this — it's new) |
| **Guardrail** | Validation that runs alongside the agent | Your `max_steps` plus any input checks |
| **Tracing** | Built-in observability of agent runs | Your `trace` list |

You'll notice that **every primitive except Handoffs maps directly to something you wrote yourself**. <br>
That's not a coincidence — it's the whole reason this SDK is a good first framework. The abstractions follow the mechanics, not the other way around.

### Agents SDK or the Responses API?

A reasonable question: if I'm already comfortable making raw API calls (as we did in Part 1), do I even need an SDK?

The OpenAI docs themselves give a sensible answer:

- **Use the Responses API directly** when your workflow is short-lived and mainly about returning the model's response. You want to own the loop, tool dispatch, and state handling yourself.
- **Use the Agents SDK** when you want the runtime to manage turns, tool execution, guardrails, handoffs, or sessions. Your agent should operate across multiple coordinated steps.

Most real applications use both — the SDK for managed agent workflows, and direct API calls for the simpler, one-shot tasks. Knowing both is the right professional baseline.

---

## Section 2 — Installation and Setup

The SDK requires Python 3.10 or newer. Install it with `pip`:

```bash
pip install openai-agents
```

Or with `uv`, which is faster:

```bash
uv add openai-agents
```

You also need an OpenAI API key. Set it as an environment variable:

```bash
export OPENAI_API_KEY=sk-...
```

Or in a `.env` file:

```
OPENAI_API_KEY=sk-...
```

If you're using a `.env` file, also install `python-dotenv` and load it at the top of your script:

In [4]:
from dotenv import load_dotenv
load_dotenv()

True

That's the whole setup. Let's confirm everything works with the smallest possible agent.

### Hello, Agent

In [6]:
from agents import Agent, Runner

agent = Agent(
    name="Assistant",
    instructions="You are a helpful assistant.",
)

result = await Runner.run(agent, "Write a haiku about agents.")
print(result.final_output)

Silent agents move  
Across hidden wires and light  
Whispers shape new worlds


Three lines of agent code. Run it. You should see a haiku.

That's the dopamine hit. Now let's understand what just happened.

---

## Section 3 — Agents

The `Agent` class is the central object in the SDK. It represents an LLM configured for a specific purpose:

In [19]:
from agents import Agent

agent = Agent(
    name="Support Operations Agent",
    instructions="You help resolve questions about support tickets and SLA policies.",
    model="gpt-4o-mini",        # Optional, defaults to a sensible model
)

print(f"Agent Name: {agent.name}")
print(f"Agent Instructions: {agent.instructions}")
print(f"Agent Model: {agent.model}")
print(f"Agent Tools: {agent.tools}")

Agent Name: Support Operations Agent
Agent Instructions: You help resolve questions about support tickets and SLA policies.
Agent Model: gpt-4o-mini
Agent Tools: []


Three parameters, only two of which are required:

- **`name`** — used in tracing and logging. Make it descriptive; future-you will thank present-you.
- **`instructions`** — the system prompt. This defines the agent's behaviour, tone, and constraints.
- **`model`** — which LLM to use. Optional. Use the most capable model while prototyping, then optimise.

That's the entire mental model. An `Agent` is a configured persona that knows how to respond. Tools, structured outputs, handoffs, and guardrails are all added on top of this base object as keyword arguments — we'll get to each of them in turn.

### Instructions Matter More Than You Think

In Part 1, we wrote prompts. In the SDK, those prompts live inside `instructions`. <br>
The difference is that the SDK's runner is calling the model many times during a single agent run — once per step in the loop — so `instructions` are sent every time. <br> 
A good `instructions` block doesn't just describe the agent's role; it also gives the agent the patterns it should follow when deciding what to do next.

For our support agent, a much stronger version of `instructions` looks like this:

In [11]:
INSTRUCTIONS = """
You are a support operations assistant. You help users understand the status
of support tickets and the SLA policies that apply to them.

When asked about a ticket:
1. First look up the ticket using get_ticket.
2. Then look up the SLA policy for that ticket's customer using get_sla_policy.
3. Combine the two pieces of information into a clear, single answer.

Always state the SLA tier explicitly in your final answer.
If a ticket is not found, say so clearly and do not invent details.
"""

This is the same pattern we used in Part 1's `TOOL_DESCRIPTIONS` — guiding the agent with clear, explicit steps. <br>
The OpenAI guide *A Practical Guide to Building Agents* calls this **prompting agents to break down tasks**, and it remains one of the highest-leverage things you can do to improve agent reliability.


---

## Section 4 — Tools

In Part 1, we did three things to give the model tools:

1. Defined Python functions
2. Registered them in a `TOOLS` dictionary
3. Wrote a long block of text describing them, the calling convention, and the expected JSON format

The SDK collapses all three of those steps into a single decorator.

In [12]:
from agents import function_tool

@function_tool
def get_ticket(ticket_id: str) -> str:
    """Look up a support ticket by ID. Returns subject, customer ID and status."""
    ticket = TICKETS.get(ticket_id.upper())
    if not ticket:
        return f"No ticket found with ID: {ticket_id}"
    return (
        f"Subject: {ticket['subject']}, "
        f"Customer: {ticket['customer_id']}, "
        f"Status: {ticket['status']}"
    )

That's it. The `@function_tool` decorator:

- Reads the function's **type hints** and turns them into a JSON schema for the model
- Reads the **docstring** and uses it as the tool description
- Registers the function so the runner can dispatch it when the model calls it

This is what we wrote by hand in Part 1, automated. The `TOOLS` dictionary, the `TOOL_DESCRIPTIONS` text, and the JSON dispatch logic are all collapsed into one decorator.

### Why Type Hints and Docstrings Matter

Look at the decorated function above. Every detail in it carries weight:


In [13]:
@function_tool                                              # ← registration
def get_ticket(ticket_id: str) -> str:                      # ← input/output schema
    """Look up a support ticket by ID. ..."""               # ← description for the model
    ...                                                     # ← actual logic

A function with no docstring is a function the model can't use confidently. A function with a vague docstring is a function the model uses *unreliably*. Treat docstrings as agent UX — they are the only interface the model has to your tools.

### Adding Tools to an Agent

Once defined, tools are added to an agent via the `tools` parameter:

In [15]:
support_agent = Agent(
    name="Support Operations Agent",
    instructions=INSTRUCTIONS,
    tools=[get_ticket],
)

That's the full equivalent of what we did in Part 1, where we registered tools in a dictionary and then described them in a multi-line `TOOL_DESCRIPTIONS` string. Same outcome, fraction of the code.


---

## Section 5 — The Runner

The `Runner` is the loop you no longer have to write.

In Part 1, our `run_agent()` function looked roughly like this:

```python
for step in range(1, max_steps + 1):
    decision = decide_next_step(...)
    if decision.action == "answer":
        return ...
    if decision.action == "use_tool":
        result = TOOLS[decision.tool_name](decision.tool_input)
        observations.append(...)
```

The SDK's `Runner` does exactly that, but with proper error handling, retries, parallel tool execution, and tracing built in. You call it like this:

```python
import asyncio
from agents import Runner

async def main():
    result = await Runner.run(support_agent, "What is the SLA for ticket T-202?")
    print(result.final_output)

asyncio.run(main())
```

Three ways to call it, depending on context:

| Method | When to use |
|---|---|
| `Runner.run(agent, input)` | Async — preferred for production code |
| `Runner.run_sync(agent, input)` | Synchronous wrapper — convenient in scripts and notebooks |
| `Runner.run_streamed(agent, input)` | Async + streams events as they happen — for live UIs |

In a Jupyter notebook, top-level `await` works directly:

```python
result = await Runner.run(support_agent, "What is the SLA for ticket T-202?")
print(result.final_output)
```

No `asyncio.run()` wrapper needed. If you're new to async Python, the only thing to remember is: `await` it, or use `run_sync`.

### The RunResult

`Runner.run()` returns a `RunResult` object that contains everything about what happened during the run:

```python
result = await Runner.run(support_agent, "What is the SLA for ticket T-202?")

print(result.final_output)        # The final answer string
print(result.last_agent.name)     # Which agent produced it (matters with handoffs)
print(result.new_items)           # Every tool call, response, and handoff in order
print(result.to_input_list())     # The full message list, ready for the next turn
```

If you ever need to manually inspect what happened — which tool was called, what it returned, how many turns were used — the `RunResult` is where that information lives. It's the SDK's structured equivalent of the `trace` list you built in Part 1.

---

## Section 6 — Structured Outputs

In Part 1, we used Pydantic to validate the model's JSON output. The SDK has this built in via the `output_type` parameter.

Suppose we want our support agent to return a structured analysis, not just prose:

```python
from pydantic import BaseModel
from agents import Agent

class TicketAnswer(BaseModel):
    ticket_id: str
    customer_name: str
    sla_tier: str
    response_sla: str
    resolution_sla: str
    summary: str

support_agent = Agent(
    name="Support Operations Agent",
    instructions=INSTRUCTIONS,
    tools=[get_ticket, get_sla_policy],
    output_type=TicketAnswer,
)
```

Now when you run the agent, `result.final_output` is no longer a string — it's a validated `TicketAnswer` object:

```python
result = await Runner.run(support_agent, "What is the SLA for ticket T-202?")

print(result.final_output.ticket_id)         # "T-202"
print(result.final_output.sla_tier)          # "startup"
print(result.final_output.response_sla)      # "8 hours"
```

This is the same pattern as `RequestAnalysis` from Part 1, but the SDK now handles all the JSON parsing, validation, and retry-on-failure for you. Failed validations trigger automatic retries with the error fed back to the model, so transient JSON formatting issues mostly disappear.

> **Structured outputs are how you make agents reliable enough to use in pipelines, not just chats.**


---

## Section 7 — Sessions and Memory

In Part 1, we built memory by maintaining a `history` list outside the function and passing it in on every call. <br>
The SDK formalises this with `Session` objects that persist conversation state automatically.

The simplest session is `SQLiteSession`, which stores conversation history in a local SQLite database:

```python
from agents import Agent, Runner, SQLiteSession

session = SQLiteSession("user_123_conversation", "conversations.db")

# First turn
result1 = await Runner.run(
    support_agent,
    "What is the SLA for ticket T-202?",
    session=session,
)
print(result1.final_output)

# Second turn — the agent remembers the first
result2 = await Runner.run(
    support_agent,
    "And what about ticket T-303 — does the same SLA apply?",
    session=session,
)
print(result2.final_output)
```

The session handles everything we did manually in `chat_once`:

- Appending the user message to history
- Sending the full history to the model
- Appending the agent's response to history
- Persisting it across turns

The conversation key (`"user_123_conversation"`) lets you keep separate conversations for different users in the same database. <br>
In production, you'd typically derive this from your user ID or session ID.

### Session Backends

The SDK ships with several session backends suitable for different deployments:

| Backend | Use case |
|---|---|
| `SQLiteSession` | Local development, small deployments, single-process apps |
| `SQLAlchemySession` | Production-grade with Postgres or other SQL databases |
| `RedisSession` | Distributed deployments needing fast shared state |
| `MongoDBSession` | NoSQL backends |
| `EncryptedSession` | When conversation content is sensitive |

For Day 1, you'll only need `SQLiteSession`. We'll go much deeper into memory design on Day 3.

---

## Section 8 — Handoffs


Handoffs are the one primitive that we did *not* build in Part 1. They're worth understanding because they unlock multi-agent design without much extra complexity.

The idea is simple: an agent can delegate the rest of a task to another, more specialised agent. The conversation transfers cleanly, and the new agent takes over until it either finishes or hands off again.

```
              ┌─────────────┐
   user ───▶  │  Triage     │
              │  Agent      │
              └─────┬───────┘
                    │
       ┌────────────┼─────────────┐
       ▼            ▼             ▼
 ┌──────────┐ ┌──────────┐ ┌──────────┐
 │ Support  │ │  Sales   │ │ Billing  │
 │  Agent   │ │  Agent   │ │  Agent   │
 └──────────┘ └──────────┘ └──────────┘
```

### A Worked Example

Suppose our support agent should hand off billing questions to a billing specialist. We define two agents:

```python
billing_agent = Agent(
    name="Billing Specialist",
    handoff_description="Specialist for invoice, payment, and refund questions.",
    instructions=(
        "You handle billing-related issues. Answer questions about invoices, "
        "payments, refunds, and pricing. If the user's question is not about "
        "billing, hand back to the support agent."
    ),
    tools=[get_invoice, get_refund_policy],
)

support_agent = Agent(
    name="Support Operations Agent",
    instructions=(
        "You help with support tickets and SLA policies. "
        "If the question is about billing, payments, or refunds, "
        "hand off to the Billing Specialist."
    ),
    tools=[get_ticket, get_sla_policy],
    handoffs=[billing_agent],
)
```

Now when a user asks the support agent about a refund, the runner will automatically route the conversation to `billing_agent`:

```python
result = await Runner.run(
    support_agent,
    "I want a refund for invoice INV-0042.",
)

print(result.final_output)
print(f"Answered by: {result.last_agent.name}")
# Answered by: Billing Specialist
```

### Handoffs vs Agents-as-Tools

There are two ways to coordinate multiple agents in the SDK, and they're easy to confuse. Both are useful, but they have different semantics:

- **Handoffs** — the specialist *takes over* the conversation. The original agent steps out of the loop.
- **Agents as tools** — the original agent stays in control and *calls* the specialist like a function, then synthesises the result.

We'll go deeper into multi-agent patterns on Day 4. For now, the takeaway is: handoffs exist as a first-class primitive, and you trigger them just by listing them in `handoffs=[...]`.

---

## Section 9 — Guardrails


In Part 1, we built one guardrail: a `max_steps` limit to prevent runaway loops. <br>
The SDK supports a broader, first-class concept of guardrails as **validation checks that run alongside the agent**.

There are two types:

- **Input guardrails** — run on the user's input before the agent processes it. Used for things like blocking off-topic queries or detecting prompt injection.
- **Output guardrails** — run on the agent's final output before it's returned. Used for things like PII detection, brand-safety checks, or output schema validation.

Both run *concurrently* with the agent. If a guardrail trips, the SDK raises a tripwire exception and you handle it however your application requires.

### Example: An Input Guardrail

Suppose we want to block clearly off-topic queries from reaching our support agent. The pattern uses a small "checker" agent that returns a structured verdict:

```python
from pydantic import BaseModel
from agents import (
    Agent, Runner,
    GuardrailFunctionOutput, input_guardrail,
    RunContextWrapper, TResponseInputItem,
)

class TopicCheck(BaseModel):
    is_on_topic: bool
    reasoning: str

topic_checker = Agent(
    name="Topic Checker",
    instructions=(
        "Decide if a user message is about support tickets, SLAs, billing, "
        "or refunds. Reply with on-topic = true if so, otherwise false."
    ),
    output_type=TopicCheck,
)

@input_guardrail
async def topic_guardrail(
    ctx: RunContextWrapper[None],
    agent: Agent,
    input: str | list[TResponseInputItem],
) -> GuardrailFunctionOutput:
    result = await Runner.run(topic_checker, input, context=ctx.context)
    return GuardrailFunctionOutput(
        output_info=result.final_output,
        tripwire_triggered=not result.final_output.is_on_topic,
    )

support_agent = Agent(
    name="Support Operations Agent",
    instructions=INSTRUCTIONS,
    tools=[get_ticket, get_sla_policy],
    input_guardrails=[topic_guardrail],
)
```

Now if someone asks "what's the capital of France?", the guardrail trips before the main agent ever runs:

```python
from agents import InputGuardrailTripwireTriggered

try:
    await Runner.run(support_agent, "What is the capital of France?")
except InputGuardrailTripwireTriggered:
    print("Off-topic query blocked.")
```

### Why Guardrails Run Concurrently

In a naive design, you'd run the guardrail first, wait for it, then run the main agent. The SDK runs them in parallel and short-circuits if the guardrail trips. This saves latency in the common case (where most inputs are fine) without sacrificing safety in the bad case.

The OpenAI guide describes this as **optimistic execution**, and it's a small but smart piece of design.

### When to Add Guardrails

The OpenAI guide recommends starting with:

1. **Relevance classifiers** — keep the agent in-scope
2. **Safety classifiers** — detect prompt injection and jailbreaks
3. **Output validation** — make sure final outputs match your contract

Then layering more guardrails as you encounter real-world failures. We'll go deeper into this on Day 4.


---

## Section 10 — Tracing


In Part 1, we built our own trace list — a list of dictionaries recording every step, decision, and observation. It was a useful tool, but limited: it only existed during a single run, and inspecting it meant reading raw Python output.

The SDK has **built-in tracing** that captures every tool call, model response, guardrail event, and handoff automatically. By default, traces are sent to the OpenAI dashboard, where you can view them as visual timelines.

You don't have to enable anything — tracing is on by default for every `Runner.run()`:

```python
result = await Runner.run(support_agent, "What is the SLA for ticket T-202?")
```

Then go to the [OpenAI Traces dashboard](https://platform.openai.com/traces) and you'll see something like this:

```
Trace: Runner.run [3.2s]
  ├── Agent: Support Operations Agent [3.0s]
  │   ├── LLM call #1 [0.8s]    "Decide next step"
  │   ├── Tool: get_ticket("T-202") [0.1s]
  │   ├── LLM call #2 [0.9s]    "Decide next step"
  │   ├── Tool: get_sla_policy("C-14") [0.1s]
  │   └── LLM call #3 [1.1s]    "Final answer"
  └── Total tokens: 1,247
```

Every span is clickable, every input and output is inspectable, and the timeline view makes it immediately obvious where time and tokens were spent.

### Custom Trace Processors

If you want traces sent somewhere other than the OpenAI dashboard — Langfuse, Sentry, your own database — you can attach a custom processor. We'll do exactly this on Day 4 when we wire the agent up to Langfuse for production observability.

For now, the takeaway is: **you no longer need to build your own trace list**. The SDK does it for you, and it does it well.


---

## Section 11 — The Complete Support Agent (SDK Version)

Time to put it all together. Here is the full support operations agent from Part 1 — now rewritten using the SDK, with sessions, structured outputs, a guardrail, and built-in tracing.

In [20]:
import os
from dotenv import load_dotenv
from pydantic import BaseModel
from agents import (
    Agent, Runner, function_tool, SQLiteSession,
    GuardrailFunctionOutput, input_guardrail,
    RunContextWrapper, TResponseInputItem,
)

load_dotenv()

# ---------------------------------------------------------------------------
# Mock databases (same as Part 1)
# ---------------------------------------------------------------------------

TICKETS = {
    "T-101": {"subject": "Login failure",          "customer_id": "C-09", "status": "open"},
    "T-202": {"subject": "Payment not processed",  "customer_id": "C-14", "status": "open"},
    "T-303": {"subject": "Dashboard not loading",  "customer_id": "C-09", "status": "resolved"},
    "T-404": {"subject": "Export feature broken",  "customer_id": "C-21", "status": "open"},
}

CUSTOMERS = {
    "C-09": {"name": "Acme Corp",       "tier": "enterprise"},
    "C-14": {"name": "Bright Ltd",      "tier": "startup"},
    "C-21": {"name": "Nova Solutions",  "tier": "enterprise"},
}

SLA_POLICIES = {
    "enterprise": "4-hour response, 24-hour resolution",
    "startup":    "8-hour response, 72-hour resolution",
    "free":       "best effort, no guaranteed resolution time",
}

# ---------------------------------------------------------------------------
# Tools — same Python functions, now decorated
# ---------------------------------------------------------------------------

@function_tool
def get_ticket(ticket_id: str) -> str:
    """Look up a support ticket by ID. Returns subject, customer ID and status."""
    ticket = TICKETS.get(ticket_id.upper())
    if not ticket:
        return f"No ticket found with ID: {ticket_id}"
    return (
        f"Subject: {ticket['subject']}, "
        f"Customer: {ticket['customer_id']}, "
        f"Status: {ticket['status']}"
    )

@function_tool
def get_sla_policy(customer_id: str) -> str:
    """Look up the SLA policy for a customer based on their tier."""
    customer = CUSTOMERS.get(customer_id.upper())
    if not customer:
        return f"No customer found with ID: {customer_id}"
    tier   = customer["tier"]
    policy = SLA_POLICIES.get(tier, "No policy found")
    return f"{customer['name']} is on the {tier} tier: {policy}"

# ---------------------------------------------------------------------------
# Structured output
# ---------------------------------------------------------------------------

class TicketAnswer(BaseModel):
    ticket_id: str
    customer_name: str
    sla_tier: str
    summary: str

# ---------------------------------------------------------------------------
# Input guardrail — block clearly off-topic queries
# ---------------------------------------------------------------------------

class TopicCheck(BaseModel):
    is_on_topic: bool
    reasoning: str

topic_checker = Agent(
    name="Topic Checker",
    instructions=(
        "Decide if a user message is about support tickets, SLAs, customers, "
        "or related operational questions. Set is_on_topic = true if so."
    ),
    output_type=TopicCheck,
)

@input_guardrail
async def topic_guardrail(
    ctx: RunContextWrapper[None],
    agent: Agent,
    input: str | list[TResponseInputItem],
) -> GuardrailFunctionOutput:
    result = await Runner.run(topic_checker, input, context=ctx.context)
    return GuardrailFunctionOutput(
        output_info=result.final_output,
        tripwire_triggered=not result.final_output.is_on_topic,
    )

# ---------------------------------------------------------------------------
# The agent itself
# ---------------------------------------------------------------------------

INSTRUCTIONS = """
You are a support operations assistant. You help users understand the status
of support tickets and the SLA policies that apply to them.

When asked about a ticket:
1. First look up the ticket using get_ticket.
2. Then look up the SLA policy for that ticket's customer using get_sla_policy.
3. Combine the two pieces of information into a clear summary.

If a ticket is not found, say so clearly. Do not invent details.
"""

support_agent = Agent(
    name="Support Operations Agent",
    instructions=INSTRUCTIONS,
    tools=[get_ticket, get_sla_policy],
    input_guardrails=[topic_guardrail],
    output_type=TicketAnswer,
)

# ---------------------------------------------------------------------------
# Run it
# ---------------------------------------------------------------------------

async def main():
    session = SQLiteSession("demo_user", "conversations.db")

    queries = [
        "What is the SLA for ticket T-202?",
        "And ticket T-404 — what does the SLA look like for that one?",
    ]

    for query in queries:
        print(f"\nQuery: {query}")
        result = await Runner.run(support_agent, query, session=session)
        answer = result.final_output

        print(f"  Ticket:   {answer.ticket_id}")
        print(f"  Customer: {answer.customer_name}")
        print(f"  Tier:     {answer.sla_tier}")
        print(f"  Summary:  {answer.summary}")

# call it
await main()

# Calling it in regualr .py script
# if __name__ == "__main__":
#     import asyncio
#     asyncio.run(main())


Query: What is the SLA for ticket T-202?
  Ticket:   T-202
  Customer: Bright Ltd
  Tier:     startup
  Summary:  Ticket T-202 is open. Bright Ltd is on the startup tier with an 8-hour response SLA and 72-hour resolution SLA.

Query: And ticket T-404 — what does the SLA look like for that one?
  Ticket:   T-404
  Customer: Nova Solutions
  Tier:     enterprise
  Summary:  Ticket T-404 is open. Nova Solutions is on the enterprise tier with a 4-hour response SLA and 24-hour resolution SLA.


### What Just Happened

Compare the structure of this file to Part 1's `run_agent()`. Every piece of plumbing we wrote by hand is gone:

- No `decide_next_step()` function — `Runner.run()` calls the model for us
- No `TOOLS` dictionary — `@function_tool` registers them automatically
- No `TOOL_DESCRIPTIONS` block — generated from docstrings and type hints
- No `AgentDecision` Pydantic model — `TicketAnswer` is the *output* type, not an internal control structure
- No `max_steps` guardrail or `trace` list — both are built into the runner
- No `history` list management — `SQLiteSession` handles it

But the **mechanics** are identical. The runner still calls the model, parses its decision, dispatches a tool, appends the result as an observation, and loops. The guardrail still enforces boundaries. The trace still gets recorded.

You haven't lost any understanding. You've gained a production-grade implementation of the agent you already knew how to build.


---

## Section 12 — Where to Go Next


You now know enough of the OpenAI Agents SDK to start building real agents. Let's set expectations for what's ahead.

### What the SDK Gave You for Free

| You no longer have to write | The SDK provides |
|---|---|
| The agent loop | `Runner.run()` |
| Tool registration and dispatch | `@function_tool` |
| Structured output parsing and retry | `output_type=` |
| Conversation memory and persistence | `Session` classes |
| Multi-agent coordination | `handoffs=[...]` |
| Input/output validation | `input_guardrails`, `output_guardrails` |
| Tracing and observability | Automatic, with dashboard view |
| Async execution, streaming, parallel tool calls | All built in |

### What You Still Have to Design

The framework cannot make these decisions for you:

- **What tools to give the agent** — schema, semantics, error handling
- **What the instructions should say** — the agent's behaviour lives here
- **What guardrails matter for your use case** — relevance, safety, PII, brand
- **What the right output schema is** — structured outputs are only as good as their design
- **What evaluation looks like** — how do you know the agent is working?

These are the genuine engineering decisions in any agent system. The SDK is the easy part. The design is the hard part. The rest of this course is about getting good at the design.

### Other Frameworks Worth Knowing

The SDK we're using is OpenAI's, but the patterns you learned here transfer cleanly to other frameworks:

- **Pydantic AI** — close to the metal, model-agnostic, very thin abstractions. Excellent if you want to stay vendor-neutral.
- **LangGraph** — graph-based orchestration. More opinionated, but powerful for complex multi-agent workflows.
- **Anthropic's tool use API** — Claude's native tool calling. Different SDK, same underlying pattern.

In all of these, you will recognise the agent, the tools, the loop, the observations, the trace, and the guardrails — because you built them yourself first. The frameworks are not magic. They are your `run_agent()` function, written by someone who has handled more edge cases than you have.

### What's Coming on Days 2–5

This is Day 1 of a five-day course. Here's where each primitive you just learned gets deepened:

| Day | Theme | What you'll deepen |
|---|---|---|
| **Day 2** | Tools, APIs, Files, Structured Outputs | `@function_tool` with real APIs, schema validation, files |
| **Day 3** | Context Engineering and Memory | `Session` in depth, Mem0 integration, context window management |
| **Day 4** | Multi-Agent Workflows, Observability, Evaluation | Handoffs, agents-as-tools, Langfuse, eval frameworks |
| **Day 5** | Capstone Build | All of the above, applied to a real project |

You now have the foundation for all of it. Welcome to professional AI engineering.

---

## Recap: The Twelve Primitives

| Primitive | Lives In | What It Replaces From Part 1 |
|---|---|---|
| `Agent` | `from agents import Agent` | Your system prompt + tool list |
| `function_tool` | `from agents import function_tool` | The `TOOLS` dictionary and `TOOL_DESCRIPTIONS` text |
| `Runner.run()` | `from agents import Runner` | The `run_agent()` while loop |
| `RunResult` | Returned from `Runner.run()` | The dictionary returned by `run_agent()` |
| `output_type=` | `Agent(..., output_type=Model)` | Manual Pydantic JSON parsing |
| `SQLiteSession` | `from agents import SQLiteSession` | The `history` list passed to `chat_once()` |
| `handoffs=[...]` | `Agent(..., handoffs=[other])` | (New — not built in Part 1) |
| `@input_guardrail` | `from agents import input_guardrail` | The `max_steps` check and any input checks |
| `output_guardrails=[...]` | `Agent(..., output_guardrails=[...])` | (New — not built in Part 1) |
| Built-in tracing | Automatic | The `trace` list in `run_agent()` |
| `Runner.run_streamed()` | For streaming UIs | (New — not built in Part 1) |
| Multi-agent orchestration | Combine all of the above | (New — not built in Part 1) |

That's it. Twelve primitives, mapping directly to the mechanics you already understand.

---

*If you found this useful, the code from this article is available as a runnable Jupyter notebook in the course repository. Feedback, corrections, and pull requests are very welcome.*
